# Fashion-MNIST CNN — Optuna & Hyperopt

**Task:** Build a CNN classifier for Fashion-MNIST, then use **Optuna** and **Hyperopt** to find strong hyperparameters and train a final model with the selected configuration.

In [ ]:
# Install if needed:
# !pip install -q tensorflow optuna hyperopt pandas scikit-learn matplotlib

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1. Load and prepare Fashion-MNIST

Fashion-MNIST contains 60,000 training images and 10,000 test images, each 28×28 grayscale with 10 classes.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train_full = np.expand_dims(x_train_full, -1)
x_test = np.expand_dims(x_test, -1)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_full
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

In [ ]:
plt.figure(figsize=(8,4))
for i in range(10):
    ax = plt.subplot(2,5,i+1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Baseline CNN

In [ ]:
def build_baseline_cnn():
    model = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline = build_baseline_cnn()
baseline_history = baseline.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=64,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
)

baseline_test_loss, baseline_test_acc = baseline.evaluate(x_test, y_test, verbose=0)
print("Baseline test accuracy:", baseline_test_acc)

## 3. Optuna tuning

Tune convolution filters, dense units, dropout, learning rate, and batch size.

In [ ]:
def build_fashion_model(params):
    model = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(params["filters1"], 3, activation=params["activation"]),
        layers.MaxPooling2D(),
        layers.Conv2D(params["filters2"], 3, activation=params["activation"]),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(params["dense_units"], activation=params["activation"]),
        layers.Dropout(params["dropout"]),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(params["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def fashion_optuna_objective(trial):
    tf.keras.backend.clear_session()

    params = {
        "filters1": trial.suggest_categorical("filters1", [16, 32, 48]),
        "filters2": trial.suggest_categorical("filters2", [32, 64, 96]),
        "dense_units": trial.suggest_categorical("dense_units", [64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5, step=0.1),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "activation": trial.suggest_categorical("activation", ["relu", "elu"]),
    }
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    model = build_fashion_model(params)
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=12,
        batch_size=batch_size,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
    )
    return float(max(history.history["val_accuracy"]))

optuna_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
optuna_study.optimize(fashion_optuna_objective, n_trials=25, show_progress_bar=False)

print("Optuna best validation accuracy:", optuna_study.best_value)
print("Optuna best parameters:")
print(optuna_study.best_params)

In [ ]:
display(pd.DataFrame(optuna_study.trials_dataframe()).sort_values("value", ascending=False).head(10))

## 4. Hyperopt tuning

In [ ]:
def hyperopt_fashion_objective(params):
    tf.keras.backend.clear_session()

    model = build_fashion_model({
        "filters1": int(params["filters1"]),
        "filters2": int(params["filters2"]),
        "dense_units": int(params["dense_units"]),
        "dropout": float(params["dropout"]),
        "learning_rate": float(params["learning_rate"]),
        "activation": params["activation"],
    })

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=12,
        batch_size=int(params["batch_size"]),
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
    )

    best_acc = float(max(history.history["val_accuracy"]))
    return {"loss": -best_acc, "status": STATUS_OK}

space = {
    "filters1": hp.choice("filters1", [16, 32, 48]),
    "filters2": hp.choice("filters2", [32, 64, 96]),
    "dense_units": hp.choice("dense_units", [64, 128, 256]),
    "dropout": hp.quniform("dropout", 0.1, 0.5, 0.1),
    "learning_rate": hp.loguniform("learning_rate", np.log(1e-4), np.log(3e-3)),
    "activation": hp.choice("activation", ["relu", "elu"]),
    "batch_size": hp.choice("batch_size", [32, 64, 128]),
}

hyperopt_trials = Trials()
best_hyperopt = fmin(
    fn=hyperopt_fashion_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=25,
    trials=hyperopt_trials,
    rstate=np.random.default_rng(SEED)
)

best_hyperopt_score = max(-t["result"]["loss"] for t in hyperopt_trials.trials)
print("Hyperopt best validation accuracy:", best_hyperopt_score)
print("Hyperopt raw best result:")
print(best_hyperopt)

## 5. Final model using Optuna best parameters

In [ ]:
best = optuna_study.best_params.copy()
batch_size = best.pop("batch_size")

final_model = build_fashion_model(best)
final_history = final_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=25,
    batch_size=batch_size,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
)

test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print("Final test accuracy:", test_acc)
print("Final test loss:", test_loss)

pred = np.argmax(final_model.predict(x_test, verbose=0), axis=1)
print(classification_report(y_test, pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(8,7))
plt.imshow(cm)
plt.title("Fashion-MNIST Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
plt.show()

plt.figure(figsize=(7,4))
plt.plot(final_history.history["accuracy"], label="train")
plt.plot(final_history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Final Model Accuracy")
plt.legend()
plt.show()

In [ ]:
with open("fashion_mnist_best_params_optuna.json", "w") as f:
    json.dump(optuna_study.best_params, f, indent=2)

final_model.save("fashion_mnist_final_optuna.keras")

print("Saved: fashion_mnist_best_params_optuna.json")
print("Saved: fashion_mnist_final_optuna.keras")

### Final takeaway

This notebook compares a baseline CNN with tuned training using Optuna and Hyperopt. The final test set is kept separate from the tuning process.

**Important:** exact best parameters and accuracy are produced by executing the notebook; they should not be hard-coded because tuning results depend on the run.